In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # fixing crashing kernel
from faster_whisper import WhisperModel
from pathlib import Path
from tqdm import tqdm


MODEL_SIZE = "medium"
DEVICE = "cuda"
COMPUTE_TYPE = "float16" 

INPUT_FOLDER = Path(r"d:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW")
OUTPUT_FOLDER = Path(r"d:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW")

print(f"Loading Whisper model '{MODEL_SIZE}'...")
model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

wav_files = sorted(INPUT_FOLDER.glob("*.wav"))

for wav_path in tqdm(wav_files, desc="Transcribing"):
    try:
        segments, info = model.transcribe(
            str(wav_path),
            language="en",
            hotwords="Uh", # nudge model towards this word when in doubt
            beam_size=3, # allow for more possibilities 
            temperature=0.0, # documentation states this should just be 0.0
            suppress_tokens=[], # tells model to not supress any tokens, thus allowing it to transcribe disfluencies and repetitions
        )

        transcript = " ".join(seg.text for seg in segments).strip()

        out_file = OUTPUT_FOLDER / f"{wav_path.stem}.txt"
        with open(out_file, "w", encoding="utf-8") as f:
            f.write(transcript)

    except Exception as e:
        print(f"Error processing {wav_path.name}: {e}")

print("Done")


Loading Whisper model 'medium'...


Transcribing: 100%|██████████| 6866/6866 [3:04:57<00:00,  1.62s/it]  

Done


### Checks for possible bugged transcriptions

In [ ]:
import re
from pathlib import Path
from collections import Counter

INPUT_FOLDER = Path(r"D:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW")

MIN_WORDS = 3
DOM_SENTENCE_RATIO = 0.6
DOM_WORD_RATIO = 0.5
MIN_LEXICAL_DIVERSITY = 0.15
MAX_BRACKET_RATIO = 0.4

DISFLUENCY_REGEX = re.compile(
    r"\b(uh+|um+|uhm+|erm|er|ah+|hm+|hmm+)\b|\[(uh+|um+|uhm+|silence|blank_audio)\]",
    re.IGNORECASE,
)


def split_sentences(text):
    return [
        s.strip()
        for s in re.split(r"(?:\s{2,}|(?<=[?.!,]))", text)
        if s.strip()
    ]

def tokenize_words(text):
    return [w for w in text.split(" ") if w.strip()]

def bracket_tokens(words):
    return [w for w in words if w.startswith("[") and w.endswith("]")]


flagged = []

for txt_path in INPUT_FOLDER.glob("*.txt"):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    reasons = []

    words = tokenize_words(text)
    sentences = split_sentences(text)

    total_words = len(words)
    total_sentences = len(sentences)

    # 1. Empty / very short
    if total_words < MIN_WORDS:
        reasons.append("short_file")

    # 2. Dominant sentence loop
    #if total_sentences > 0:
    #    sent_counts = Counter(sentences)
    #    most_common_sent, sent_freq = sent_counts.most_common(1)[0]
    #    if sent_freq / total_sentences >= DOM_SENTENCE_RATIO:
    #        reasons.append("dominant_sentence")

    # 3. Dominant word loop
    if total_words > 0:
        word_counts = Counter(words)
        most_common_word, word_freq = word_counts.most_common(1)[0]
        if word_freq / total_words >= DOM_WORD_RATIO:
            reasons.append("dominant_word")

    # 4. Lexical diversity
    if total_words > 0:
        lexical_div = len(set(words)) / total_words
        if lexical_div < MIN_LEXICAL_DIVERSITY:
            reasons.append("low_lexical_diversity")

    # 5. Excessive bracket tokens
    if total_words > 0:
        bracket_ratio = len(bracket_tokens(words)) / total_words
        if bracket_ratio > MAX_BRACKET_RATIO:
            reasons.append("excessive_bracket_tokens")

    # 6. No disfluencies detected
    #if total_words >= 50:
    #    if not DISFLUENCY_REGEX.search(text):
    #        reasons.append("no_disfluencies")

    if reasons:
        flagged.append((txt_path.name, reasons))


print("Flagged transcripts:\n")
for fname, reasons in flagged:
    print(f"{fname} | {', '.join(reasons)}")

print(f"\nTotal flagged: {len(flagged)}")

# Save flagged filenames to disk
FLAGGED_LIST_PATH = INPUT_FOLDER / "flagged_files.txt"

with open(FLAGGED_LIST_PATH, "w", encoding="utf-8") as f:
    for fname, _ in flagged:
        f.write(fname + "\n")

print(f"\nSaved flagged list to: {FLAGGED_LIST_PATH}")



Flagged transcripts:

312_011.txt | short_file, dominant_word
318_005.txt | short_file, dominant_word
333_033.txt | dominant_word
350_017.txt | dominant_word, low_lexical_diversity
364_016.txt | dominant_word
372_010.txt | dominant_word
377_033.txt | dominant_word
380_014.txt | dominant_word
381_002.txt | dominant_word
384_005.txt | dominant_word
400_007.txt | dominant_word
404_001.txt | dominant_word
408_015.txt | dominant_word
410_002.txt | dominant_word
411_005.txt | dominant_word
411_032.txt | short_file, dominant_word
413_008.txt | dominant_word
415_003.txt | dominant_word
416_018.txt | short_file, dominant_word
426_003.txt | dominant_word
442_024.txt | dominant_word
455_017.txt | dominant_word
465_020.txt | short_file, dominant_word
476_007.txt | short_file, dominant_word
487_010.txt | dominant_word
609_011.txt | dominant_word
619_001.txt | short_file, dominant_word
622_025.txt | short_file, dominant_word
624_008.txt | short_file, dominant_word
627_001.txt | dominant_word
641_009

### Files not flagged are sorted into an MFA ready structure, flagged files into a seperate folder

In [ ]:
import os
import shutil

CORPUS_DIR = r"D:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW"
FLAGGED_LIST = r"D:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW\flagged_files.txt"

# =====================
flagged_set = set()

if os.path.exists(FLAGGED_LIST):
    with open(FLAGGED_LIST, "r", encoding="utf-8") as f:
        for line in f:
            flagged_set.add(line.strip())
else:
    print("No flagged list found — continuing without filtering.")


flagged_folder = os.path.join(CORPUS_DIR, "flagged")
os.makedirs(flagged_folder, exist_ok=True)


for filename in os.listdir(CORPUS_DIR):
    if filename.endswith(".wav"):
        base_name = os.path.splitext(filename)[0]
        speaker_id = base_name.split("_")[0]

        wav_path = os.path.join(CORPUS_DIR, filename)
        txt_name = base_name + ".txt"
        txt_path = os.path.join(CORPUS_DIR, txt_name)

        if not os.path.exists(txt_path):
            print(f"Missing transcript for {filename}, skipping.")
            continue


        if txt_name in flagged_set:
            shutil.move(wav_path, os.path.join(flagged_folder, filename))
            shutil.move(txt_path, os.path.join(flagged_folder, txt_name))
            print(f"Moved {base_name} to FLAGGED")
            continue


        speaker_folder = os.path.join(CORPUS_DIR, speaker_id)
        os.makedirs(speaker_folder, exist_ok=True)

        shutil.move(wav_path, os.path.join(speaker_folder, filename))
        shutil.move(txt_path, os.path.join(speaker_folder, txt_name))

        print(f"Moved {base_name} to folder {speaker_id}")

print("Restructuring complete.")



Moved 309_001 to folder 309
Moved 309_002 to folder 309
Moved 309_003 to folder 309
Moved 309_004 to folder 309
Moved 309_005 to folder 309
Moved 309_006 to folder 309
Moved 309_007 to folder 309
Moved 309_008 to folder 309
Moved 309_009 to folder 309
Moved 311_001 to folder 311
Moved 311_002 to folder 311
Moved 311_003 to folder 311
Moved 311_004 to folder 311
Moved 311_005 to folder 311
Moved 311_006 to folder 311
Moved 311_007 to folder 311
Moved 311_008 to folder 311
Moved 311_009 to folder 311
Moved 311_010 to folder 311
Moved 311_011 to folder 311
Moved 311_012 to folder 311
Moved 311_013 to folder 311
Moved 311_014 to folder 311
Moved 312_001 to folder 312
Moved 312_002 to folder 312
Moved 312_003 to folder 312
Moved 312_004 to folder 312
Moved 312_005 to folder 312
Moved 312_006 to folder 312
Moved 312_007 to folder 312
Moved 312_008 to folder 312
Moved 312_009 to folder 312
Moved 312_010 to folder 312
Moved 312_011 to FLAGGED
Moved 312_012 to folder 312
Moved 312_013 to folder

### After alignment we move the output files into their corresponding folders

In [2]:
import os
import shutil

CORPUS_DIR = r"D:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW"
ALIGNMENT_DIR = r"D:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW\output"
for root, dirs, files in os.walk(ALIGNMENT_DIR):
    for file in files:
        if file.endswith(".TextGrid"):
            textgrid_path = os.path.join(root, file)

            # Extract speaker folder name from path
            speaker_id = os.path.basename(root)

            target_folder = os.path.join(CORPUS_DIR, speaker_id)

            if not os.path.exists(target_folder):
                print(f"Speaker folder missing: {speaker_id}")
                continue

            target_path = os.path.join(target_folder, file)

            shutil.copy2(textgrid_path, target_path)
            print(f"Copied {file} → {speaker_id}")

print("Finished copying TextGrids.")


Copied 309_001.TextGrid → 309
Copied 309_002.TextGrid → 309
Copied 309_003.TextGrid → 309
Copied 309_004.TextGrid → 309
Copied 309_005.TextGrid → 309
Copied 309_006.TextGrid → 309
Copied 309_007.TextGrid → 309
Copied 309_008.TextGrid → 309
Copied 309_009.TextGrid → 309
Copied 311_001.TextGrid → 311
Copied 311_002.TextGrid → 311
Copied 311_003.TextGrid → 311
Copied 311_004.TextGrid → 311
Copied 311_005.TextGrid → 311
Copied 311_006.TextGrid → 311
Copied 311_007.TextGrid → 311
Copied 311_008.TextGrid → 311
Copied 311_009.TextGrid → 311
Copied 311_010.TextGrid → 311
Copied 311_011.TextGrid → 311
Copied 311_012.TextGrid → 311
Copied 311_013.TextGrid → 311
Copied 311_014.TextGrid → 311
Copied 312_001.TextGrid → 312
Copied 312_002.TextGrid → 312
Copied 312_003.TextGrid → 312
Copied 312_004.TextGrid → 312
Copied 312_005.TextGrid → 312
Copied 312_006.TextGrid → 312
Copied 312_007.TextGrid → 312
Copied 312_008.TextGrid → 312
Copied 312_009.TextGrid → 312
Copied 312_010.TextGrid → 312
Copied 312

### Then we check if any files didn't get aligned

In [4]:
import os
import csv

CORPUS_DIR = r"D:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW"
REPORT_FILE = "missing_textgrids.csv"

missing = []
count = 0
for speaker_id in os.listdir(CORPUS_DIR):
    speaker_path = os.path.join(CORPUS_DIR, speaker_id)

    if not os.path.isdir(speaker_path):
        continue

    for file in os.listdir(speaker_path):
        if file.endswith(".wav"):
            base = os.path.splitext(file)[0]
            textgrid_path = os.path.join(speaker_path, base + ".TextGrid")
            count += 1
            if not os.path.exists(textgrid_path):
                print(f"Missing TextGrid: {speaker_id}/{base}")
                missing.append({
                    "Speaker_ID": speaker_id,
                    "File": base
                })
                count -= 1

# Save report
with open(REPORT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["Speaker_ID", "File"])
    writer.writeheader()
    writer.writerows(missing)

print(f"\nCheck complete. Missing files: {len(missing)}")
print(f"Report saved to {REPORT_FILE}")
print(f"Total utterances alinged: {count}")


Missing TextGrid: 343/343_021
Missing TextGrid: 369/369_014
Missing TextGrid: 377/377_048
Missing TextGrid: 379/379_031
Missing TextGrid: 402/402_012
Missing TextGrid: 407/407_016
Missing TextGrid: 408/408_022
Missing TextGrid: 420/420_012
Missing TextGrid: 422/422_018
Missing TextGrid: 431/431_004
Missing TextGrid: 450/450_026
Missing TextGrid: 609/609_035
Missing TextGrid: flagged/350_017

Check complete. Missing files: 13
Report saved to missing_textgrids.csv
Total utterances alinged: 6853
